# Pokémon vs. Digimon: Why Generalization Works
## An Interactive Tour of |H|, N, and the Hoeffding Bound

Imagine a single number — the **edge count** `e` of a creature's outline — and one question: *is this creature a Pokémon or a Digimon?* The simplest possible model is a single knob: call it Digimon when `e ≥ h`. This notebook uses that toy to attack a deep question in machine learning:

> **Why do a larger training set $N$ and a smaller hypothesis set $|H|$ make a *sampled* training set trustworthy?**

We only ever see a finite, random slice of the world, yet we want guarantees about the *whole* world. When — and why — is that slice good enough?

### What you'll be able to do
- Build a threshold classifier $f_h$ and read $|H|$ as a measure of model complexity.
- Compute the 0/1 loss (error rate) and watch it change as you move the threshold.
- Tell apart the *ideal* hypothesis $h_{all}$ from the *empirically chosen* $h_{train}$, and measure the **generalization gap**.
- State the good-training-set (uniform-deviation) condition and estimate $P(D_{train}\text{ is bad})$ by Monte Carlo.
- Derive and check the distribution-free guarantee $P(D_{train}\text{ is bad}) \le |H|\cdot 2\exp(-2N\varepsilon^2)$.
- Invert the bound into the sample-complexity rule $N \ge \log(2|H|/\delta)/(2\varepsilon^2)$ and reproduce $N \ge 610$.
- Quantify the bias–complexity tradeoff that motivates deep learning.

### Roadmap

| Section | Concept | What you'll do |
|---|---|---|
| C03 | Threshold classifier & $|H|$ | Define $f_h$, the hypothesis set, and model complexity |
| C05 / C06 | Loss as error rate | Drag a threshold; sweep $H$ for the ideal $h_{all}$ |
| C07 / C08 | Empirical vs ideal risk | Sample $D_{train}$, find $h_{train}$, measure the gap |
| C09 / C10 | Good vs bad training set | State the condition; estimate $P(\text{bad})$ |
| C11 / C12 | Hoeffding + union bound | Derive the bound; compare theory vs experiment |
| C13 | Sample complexity | Read off the required $N$ |
| C14 | Complexity tradeoff | See the U-shape and the deep-learning resolution |

> ℹ️ **All data in this notebook is synthetic** — a self-contained stand-in for the original Pokémon/Digimon images — so everything runs end-to-end in Colab with no downloads.

In [ ]:
# === Environment setup (the ONLY place we import libraries and create the RNG) ===
import numpy as np
import matplotlib.pyplot as plt

# Interactive stack — verify it imports so every later widget cell is safe in Colab.
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interact_manual, IntSlider, FloatSlider, Button, Output
    _WIDGETS_OK = True
except ImportError:
    print('ipywidgets not found — run `!pip install ipywidgets` and re-run this cell.')
    widgets = None
    _WIDGETS_OK = False

from IPython.display import display, clear_output

# Inline figures in Colab/Jupyter; fall back gracefully outside IPython.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    try:
        plt.switch_backend('Agg')
    except Exception:
        pass

SEED = 7

def set_seed(seed=7):
    '''Create and return a fresh seeded numpy Generator (reseeding is deterministic).'''
    return np.random.default_rng(seed)

# The single shared RNG used across the notebook.
rng = set_seed(SEED)

# --- shared plotting helpers ---
COLORS = {'pokemon': '#f0a500', 'digimon': '#2a6f97', 'accent': '#d62828', 'ideal': '#386641'}

def new_axes(figsize=(8, 4)):
    '''Return a fresh (fig, ax).'''
    fig, ax = plt.subplots(figsize=figsize)
    return fig, ax

def style_axes(ax, title='', xlabel='', ylabel=''):
    '''Light, consistent axis styling: labels, title, soft grid, despined.'''
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold')
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25, linewidth=0.7)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

ipw_ver = widgets.__version__ if _WIDGETS_OK else 'MISSING'
print(f'numpy {np.__version__}  |  ipywidgets {ipw_ver}')
print(f'Shared RNG ready (SEED={SEED}). Setup complete.')

## The model: a one-knob threshold classifier

Recall the three-step ML recipe (here only as framing):

1. **Pick a function with unknowns** — a family of candidate models.
2. **Define a loss** — how wrong a model is.
3. **Optimize** — search the family for the lowest-loss model.

Our feature is a single number, the **edge count** $e$ of a creature's silhouette (Pokémon tend to have smoother outlines and fewer edges; Digimon more). That single number is the entire input.

### Definition

$$f_h(e) = \begin{cases} \text{Digimon} & \text{if } e \ge h \\ \text{Pokémon} & \text{if } e < h \end{cases}$$

We fix the label convention **Digimon $= 1$ (positive), Pokémon $= 0$**.

The single unknown is the threshold $h$. The **hypothesis set** is every threshold we allow:

$$H = \{1, 2, \dots, 10000\}, \qquad |H| = 10000.$$

$|H|$ — the number of candidate functions — is our measure of **model complexity**.

> **More candidate functions → easier to overfit. Why?** Hold that question; we answer it quantitatively in **C14**, after building up the loss, the generalization gap, and the Hoeffding bound.

In [ ]:
# === Build the synthetic 'universe' D_all (stand-in for ALL Pokémon & Digimon) ===
EDGE_LO, EDGE_HI = 1000, 10000
EDGE_RANGE = (EDGE_LO, EDGE_HI)

def make_universe(n_pokemon=819, n_digimon=971,
                  pk_mean=4300.0, pk_std=1900.0,
                  dg_mean=6200.0, dg_std=1900.0,
                  edge_lo=EDGE_LO, edge_hi=EDGE_HI,
                  generator=None):
    '''Draw a synthetic population of edge counts for two overlapping classes.

    Returns (e, y) with y == 1 for Digimon, y == 0 for Pokemon. Edge counts are
    clipped to [edge_lo, edge_hi] BEFORE rounding so none escape the domain.
    '''
    if generator is None:
        generator = rng
    if pk_std <= 0 or dg_std <= 0:
        raise ValueError('pk_std and dg_std must be positive.')
    pk = generator.normal(pk_mean, pk_std, size=n_pokemon)
    dg = generator.normal(dg_mean, dg_std, size=n_digimon)
    e = np.concatenate([pk, dg])
    e = np.clip(e, edge_lo, edge_hi)
    e = np.round(e).astype(int)
    y = np.concatenate([np.zeros(n_pokemon, dtype=int), np.ones(n_digimon, dtype=int)])
    perm = generator.permutation(e.shape[0])
    return e[perm], y[perm]

# Materialize the FIXED universe from a fresh seeded generator so it is independent
# of any later RNG consumption.
e_all, y_all = make_universe(generator=set_seed(SEED))

assert e_all.shape == y_all.shape and len(e_all) == 1790
assert set(np.unique(y_all)).issubset({0, 1}) and int(y_all.sum()) == 971
assert e_all.min() >= EDGE_LO and e_all.max() <= EDGE_HI and e_all.dtype.kind == 'i'

# --- static preview: overlapping class histograms ---
fig, ax = new_axes(figsize=(9, 4))
bins = np.linspace(EDGE_LO, EDGE_HI, 46)
ax.hist(e_all[y_all == 0], bins=bins, histtype='stepfilled', alpha=0.55,
        color=COLORS['pokemon'], label=f'Pokémon (n={int((y_all == 0).sum())})')
ax.hist(e_all[y_all == 1], bins=bins, histtype='stepfilled', alpha=0.55,
        color=COLORS['digimon'], label=f'Digimon (n={int((y_all == 1).sum())})')
style_axes(ax, title='The universe D_all: edge-count distributions overlap',
           xlabel='edge count e', ylabel='count')
ax.legend()
plt.show()
plt.close(fig)

print(f'Universe ready: {len(e_all)} creatures '
      f'({int((y_all==0).sum())} Pokémon, {int((y_all==1).sum())} Digimon), '
      f'edge counts in [{e_all.min()}, {e_all.max()}].')

In [ ]:
# === The model f_h and the 0/1 loss, plus a live threshold explorer ===

def f_h(e, h):
    '''Predict 1 (Digimon) when edge count e >= h, else 0 (Pokemon).'''
    return (np.asarray(e) >= h).astype(int)

def loss(h, e, y):
    '''0/1 error rate L(h, D) in [0, 1]. (Cross-entropy would be an alternative.)'''
    e = np.asarray(e); y = np.asarray(y)
    if e.size == 0:
        return float('nan')
    return float(np.mean(f_h(e, h) != y))

fig_threshold_explorer = None

def _draw_threshold(h):
    global fig_threshold_explorer
    h = int(np.clip(h, EDGE_LO, EDGE_HI))
    err = loss(h, e_all, y_all)
    fig, ax = new_axes(figsize=(9, 4.2))
    bins = np.linspace(EDGE_LO, EDGE_HI, 46)
    ax.hist(e_all[y_all == 0], bins=bins, histtype='stepfilled', alpha=0.55,
            color=COLORS['pokemon'], label='Pokémon (y=0)')
    ax.hist(e_all[y_all == 1], bins=bins, histtype='stepfilled', alpha=0.55,
            color=COLORS['digimon'], label='Digimon (y=1)')
    ax.axvspan(EDGE_LO, h, color=COLORS['pokemon'], alpha=0.08)
    ax.axvspan(h, EDGE_HI, color=COLORS['digimon'], alpha=0.08)
    ax.axvline(h, color=COLORS['accent'], lw=2.2, label=f'threshold h={h}')
    ax.text(0.02, 0.95, f'L(h, D_all) = {err:.3f}', transform=ax.transAxes,
            va='top', ha='left', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', fc='white', ec=COLORS['accent'], alpha=0.9))
    style_axes(ax, title='Drag the threshold — watch the error rate change',
               xlabel='edge count e   (predict Pokémon <-- | --> predict Digimon)',
               ylabel='count')
    ax.legend(loc='upper right')
    plt.show()
    plt.close(fig)
    fig_threshold_explorer = fig

threshold_slider = IntSlider(min=EDGE_LO, max=EDGE_HI, step=50,
                             value=int(np.mean(EDGE_RANGE)),
                             continuous_update=False, description='h')
interact(_draw_threshold, h=threshold_slider)

In [ ]:
# === Sweep the whole hypothesis set H to find the ideal hypothesis h_all ===

def sweep_loss(H_grid, e, y):
    '''L(h, D) for every h in H_grid, vectorized via cumulative class counts.

    Agrees with the scalar loss() to within floating error. O(N log N + |H_grid|).
    '''
    H_grid = np.asarray(H_grid)
    e = np.asarray(e); y = np.asarray(y)
    N = e.shape[0]
    if N == 0:
        return np.full(H_grid.shape, np.nan, dtype=float)
    e_pos = np.sort(e[y == 1])   # Digimon
    e_neg = np.sort(e[y == 0])   # Pokemon
    n_neg = e_neg.shape[0]
    fp = n_neg - np.searchsorted(e_neg, H_grid, side='left')   # actual Pokemon, e >= h
    fn = np.searchsorted(e_pos, H_grid, side='left')           # actual Digimon, e <  h
    return (fp + fn) / N

H_grid = np.arange(1, EDGE_HI + 1)                  # H = {1,...,10000}, |H| = 10000
loss_curve_all = sweep_loss(H_grid, e_all, y_all)   # L(h, D_all) for every h
h_all = int(H_grid[np.argmin(loss_curve_all)])      # smallest minimizing threshold
L_h_all = float(loss_curve_all.min())

assert np.isclose(sweep_loss(np.array([h_all]), e_all, y_all)[0], loss(h_all, e_all, y_all))
assert len(H_grid) == 10000 and loss_curve_all.shape == H_grid.shape

fig, ax = new_axes(figsize=(9, 4.2))
ax.plot(H_grid, loss_curve_all, color=COLORS['digimon'], lw=1.8)
ax.axvline(h_all, color=COLORS['ideal'], lw=2.0, ls='--')
ax.scatter([h_all], [L_h_all], color=COLORS['accent'], zorder=5)
ax.annotate(f'h_all = {h_all};  L(h_all, D_all) = {L_h_all:.3f}',
            xy=(h_all, L_h_all), xytext=(0.45, 0.6), textcoords='axes fraction',
            fontsize=11, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLORS['accent']))
style_axes(ax, title='Loss over the full hypothesis set H  (the ideal baseline)',
           xlabel='threshold h', ylabel='L(h, D_all)  =  error rate')
plt.show()
plt.close(fig)

print(f'Ideal hypothesis on the universe:  h_all = {h_all},  L(h_all, D_all) = {L_h_all:.4f}')

## The catch: we never get to see $D_{all}$

In reality the universe $D_{all}$ is unobtainable. We only get a finite, random, i.i.d. **training sample** $D_{train}$. So we do the only thing we can — minimize loss on what we *can* see:

$$h_{train} = \arg\min_{h \in H} L(h, D_{train}).$$

But what we *truly care about* is how that choice does on the whole universe, $L(h_{train}, D_{all})$. The price of learning from a sample is the

$$\textbf{generalization gap} \;=\; L(h_{train}, D_{all}) - L(h_{all}, D_{all}) \;\ge\; 0.$$

### A tale of two samples (from the lecture)

Two different draws of $N \approx 200$ can tell very different stories — approximate, seed-dependent targets you'll reproduce next:

- 🍀 **Lucky draw:** $h_{train} \approx 4727$, true loss $\approx 0.28$ — essentially as good as $h_{all}$.
- 💥 **Unlucky draw:** $h_{train} \approx 3642$, train loss $\approx 0.20$ (looks great!) but true loss $\approx 0.37$ — a big gap.

The unlucky sample *looked* easier than reality. The rest of the notebook is about controlling how often that happens.

In [ ]:
# === Watch the ideal-vs-empirical gap, draw after draw ===

def sample_train(N, generator):
    '''Draw D_train of size N i.i.d. (with replacement) from the universe.'''
    idx = generator.choice(len(e_all), size=N, replace=True)
    return e_all[idx], y_all[idx]

def best_h(H_grid, e, y):
    '''Empirical risk minimizer over H_grid: returns (h_train, L(h_train, D_train)).'''
    lv = sweep_loss(H_grid, e, y)
    j = int(np.argmin(lv))
    return int(H_grid[j]), float(lv[j])

_resample_counter = {'k': 0}
out_gap = Output()
fig_train_vs_all = None

def _show_gap(N):
    global fig_train_vs_all
    with out_gap:
        clear_output(wait=True)
        try:
            gen = np.random.default_rng(SEED + _resample_counter['k'])
            e_tr, y_tr = sample_train(int(N), gen)
            h_tr, L_tr_train = best_h(H_grid, e_tr, y_tr)
            L_tr_all = loss(h_tr, e_all, y_all)
            gap = L_tr_all - L_h_all
            lv_train = sweep_loss(H_grid, e_tr, y_tr)

            fig, ax = new_axes(figsize=(9, 4.2))
            ax.plot(H_grid, loss_curve_all, color=COLORS['digimon'], lw=1.6,
                    label='L(h, D_all)  (truth)')
            ax.plot(H_grid, lv_train, color=COLORS['pokemon'], lw=1.4, alpha=0.9,
                    label='L(h, D_train)  (what we see)')
            ax.axvline(h_all, color=COLORS['ideal'], lw=1.8, ls='--', label=f'h_all={h_all}')
            ax.axvline(h_tr, color=COLORS['accent'], lw=1.8, ls=':', label=f'h_train={h_tr}')
            style_axes(ax, title=f'D_train (N={int(N)}): empirical curve vs the truth',
                       xlabel='threshold h', ylabel='error rate')
            ax.legend(loc='upper right', fontsize=9)
            plt.show()
            plt.close(fig)
            fig_train_vs_all = fig

            print(f'  L(h_train, D_train) = {L_tr_train:.3f}   <- looks this good on the sample')
            print(f'  L(h_train, D_all)   = {L_tr_all:.3f}   <- actually this good on the universe')
            print(f'  L(h_all,   D_all)   = {L_h_all:.3f}   <- best achievable')
            print(f'  generalization gap  = {gap:+.3f}')
        except Exception as exc:
            print(f'[widget error] {type(exc).__name__}: {exc}')

n_slider = IntSlider(min=20, max=1000, step=20, value=200,
                     continuous_update=False, description='N')
resample_btn = Button(description='Resample D_train', button_style='info')

def _on_resample(_btn):
    _resample_counter['k'] += 1
    _show_gap(n_slider.value)

n_slider.observe(lambda ch: _show_gap(ch['new']), names='value')
resample_btn.on_click(_on_resample)

display(widgets.VBox([widgets.HBox([n_slider, resample_btn]), out_gap]))
_show_gap(n_slider.value)   # initial render

## From good generalization to a checkable property of the sample

We want the gap small: $L(h_{train}, D_{all}) - L(h_{all}, D_{all}) \le \delta$. That involves $h_{train}$, which depends on the random sample — awkward to control directly. The trick: ask for something *stronger* but *simpler* — that **every** hypothesis looks about the same on the sample as on the universe.

> **Good training set (uniform-deviation) condition.** $D_{train}$ is **good** if
> $$\forall h \in H:\quad |L(h, D_{train}) - L(h, D_{all})| \le \varepsilon.$$

**Why it is sufficient.** Suppose $D_{train}$ is good. Then, with $\varepsilon = \delta/2$:

$$L(h_{train}, D_{all}) \;\le\; L(h_{train}, D_{train}) + \varepsilon \;\le\; L(h_{all}, D_{train}) + \varepsilon \;\le\; L(h_{all}, D_{all}) + 2\varepsilon \;=\; L(h_{all}, D_{all}) + \delta.$$

The middle step uses that $h_{train}$ minimizes the *training* loss, so it beats $h_{all}$ on the sample. Choosing $\varepsilon = \delta/2$ delivers exactly the target gap $\delta$.

A $D_{train}$ that fails the condition — $\exists h:\ |L(h, D_{train}) - L(h, D_{all})| > \varepsilon$ — is called **bad**.

This argument is **model-agnostic, distribution-free, and loss-agnostic**: it never looked at *what* the data is, only that the sample is i.i.d. and the loss is bounded. The relation $\varepsilon = \delta/2$ is the thread tying together everything that follows.

In [ ]:
# === Estimate P(D_train is bad) by Monte Carlo ===

def _subsample_grid(H_grid, k):
    '''Pick k thresholds evenly spanning H_grid (an |H| dial for speed + intuition).'''
    k = int(max(2, min(k, len(H_grid))))
    idx = np.unique(np.linspace(0, len(H_grid) - 1, k).astype(int))
    return H_grid[idx]

def estimate_p_bad(N, eps, M, H_subgrid, generator):
    '''Fraction of M i.i.d. draws whose worst-case deviation over H_subgrid exceeds eps.

    Returns (p_bad, worst_devs). Train- and universe-side losses use the SAME sub-grid.
    '''
    lv_all_sub = sweep_loss(H_subgrid, e_all, y_all)
    worst_devs = np.empty(int(M))
    for t in range(int(M)):
        idx = generator.choice(len(e_all), size=int(N), replace=True)
        lv_tr = sweep_loss(H_subgrid, e_all[idx], y_all[idx])
        worst_devs[t] = np.max(np.abs(lv_tr - lv_all_sub))
    p_bad = float(np.mean(worst_devs > eps))
    return p_bad, worst_devs

fig_p_bad_histogram = None
p_bad_empirical = None

def _run_p_bad(N, eps, M, h_count):
    global fig_p_bad_histogram, p_bad_empirical
    if eps <= 0:
        print('eps must be > 0 (every nonzero deviation would count as bad).')
        return
    sub = _subsample_grid(H_grid, h_count)
    gen = np.random.default_rng(SEED)   # reproducible across re-runs
    p_bad, devs = estimate_p_bad(N, eps, M, sub, gen)
    p_bad_empirical = p_bad

    fig, ax = new_axes(figsize=(9, 4.2))
    ax.hist(devs, bins=30, color=COLORS['digimon'], alpha=0.75)
    ax.axvline(eps, color=COLORS['accent'], lw=2.2, label=f'eps = {eps:.2f}')
    style_axes(ax,
               title=f'Worst-case deviation over |H|={len(sub)} thresholds  (N={int(N)}, M={int(M)})',
               xlabel='max_h |L(h, D_train) - L(h, D_all)|', ylabel='count')
    ax.text(0.97, 0.95, f'P(bad) ~ {p_bad:.3f}', transform=ax.transAxes,
            va='top', ha='right', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', fc='white', ec=COLORS['accent'], alpha=0.9))
    ax.legend(loc='upper left')
    plt.show()
    plt.close(fig)
    fig_p_bad_histogram = fig
    moe = np.sqrt(p_bad * (1 - p_bad) / max(int(M), 1))
    print(f'Effective work: M x |H| x N = {int(M)} x {len(sub)} x {int(N)}.  '
          f'P(D_train is bad) ~ {p_bad:.4f}  (Monte Carlo, +/- ~{moe:.3f}).')

interact_manual(_run_p_bad,
                N=IntSlider(min=20, max=1000, step=20, value=200, description='N'),
                eps=FloatSlider(min=0.02, max=0.30, step=0.01, value=0.1, description='eps'),
                M=IntSlider(min=50, max=500, step=50, value=200, description='M'),
                h_count=IntSlider(min=10, max=2000, step=10, value=200, description='|H|'))
_run_p_bad(200, 0.1, 200, 200)   # initial render

## Why the condition holds: union bound + Hoeffding

We now bound how often a sample is bad — *without knowing the data distribution.*

**Step 1 — union bound.** D_train is bad means *at least one* $h$ deviates by more than $\varepsilon$. The probability of a union is at most the sum:

$$P(D_{train}\text{ is bad}) \;=\; P\!\left(\exists h: |L(h,D_{train}) - L(h,D_{all})| > \varepsilon\right) \;\le\; \sum_{h \in H} P\!\left(|L(h,D_{train}) - L(h,D_{all})| > \varepsilon\right).$$

**Step 2 — Hoeffding's inequality.** For a fixed $h$, the loss $L(h, D_{train})$ is the average of $N$ i.i.d. random variables each in $[0,1]$, with mean $L(h, D_{all})$. Hoeffding gives

$$P\!\left(|L(h,D_{train}) - L(h,D_{all})| > \varepsilon\right) \;\le\; 2\exp(-2N\varepsilon^2).$$

**Step 3 — sum over $H$.** Every term shares the same bound, and there are $|H|$ of them:

$$\boxed{\,P(D_{train}\text{ is bad}) \;\le\; |H| \cdot 2\exp(-2N\varepsilon^2).\,}$$

It is **distribution-free** and **model-agnostic**. And it says exactly what we hoped:

- **larger $N$** → the exponential crushes the bound;
- **smaller $|H|$** → fewer terms in the sum.

Next we put numbers on it and check it against the Monte Carlo estimate from C10.

In [ ]:
# === Theory vs experiment: the Hoeffding bound over the empirical P(bad) ===

def hoeffding_bound(N, H_size, eps):
    '''|H| * 2 * exp(-2 N eps^2). NOT clipped, so the vacuous (>1) regime stays visible.'''
    return H_size * 2.0 * np.exp(-2.0 * np.asarray(N, dtype=float) * eps**2)

N_axis = np.arange(50, 1501, 25)
_empirical_N = [100, 300, 500, 800, 1000]
fig_bound_vs_empirical = None

def _draw_bound_vs_empirical(H_size, eps):
    global fig_bound_vs_empirical
    bound = hoeffding_bound(N_axis, H_size, eps)
    sub = _subsample_grid(H_grid, min(H_size, 400))   # bounded grid for the MC overlay
    emp = []
    for n in _empirical_N:
        p, _ = estimate_p_bad(n, eps, 120, sub, np.random.default_rng(SEED))
        emp.append(p)

    fig, ax = new_axes(figsize=(9, 4.6))
    ax.plot(N_axis, np.maximum(bound, 1e-12), color=COLORS['accent'], lw=2.0,
            label=f'Hoeffding bound  (|H|={H_size})')
    ax.axhline(1.0, color='gray', lw=1.0, ls='--', label='vacuous (=1)')
    ax.scatter(_empirical_N, np.maximum(emp, 1e-12), color=COLORS['digimon'],
               zorder=5, label='empirical P(bad)')
    ax.set_yscale('log')
    style_axes(ax, title='Bound (theory) vs Monte Carlo (experiment)',
               xlabel='training size N', ylabel='P(D_train is bad)   [log scale]')
    ax.legend(loc='upper right', fontsize=9)
    plt.show()
    plt.close(fig)
    fig_bound_vs_empirical = fig

    print(f'Bound at lecture points (|H|={H_size}, eps={eps:.2f}):')
    for n in (100, 500, 1000):
        b = float(hoeffding_bound(n, H_size, eps))
        tag = '  (vacuous, >1)' if b > 1 else ''
        print(f'  N={n:>4}:  bound ~ {b:.5g}{tag}')

interact(_draw_bound_vs_empirical,
         H_size=IntSlider(min=50, max=10000, step=50, value=10000, description='|H|'),
         eps=FloatSlider(min=0.05, max=0.20, step=0.01, value=0.1, description='eps'))

In [ ]:
# === Sample complexity: invert the bound to size the training set ===
import math

def required_n(H_size, delta, eps):
    '''Smallest real-valued N the bound needs: N >= log(2|H|/delta) / (2 eps^2).'''
    return math.log(2.0 * H_size / delta) / (2.0 * eps**2)

fig_sample_complexity = None

def _draw_sample_complexity(H_size, delta, eps):
    global fig_sample_complexity
    if not (eps > 0 and 0 < delta < 1 and H_size >= 1):
        print('Need eps > 0, 0 < delta < 1, |H| >= 1.')
        return
    n_real = required_n(H_size, delta, eps)
    n_ceil = int(np.ceil(n_real))
    print(f'|H|={H_size}, delta={delta:.2f}, eps={eps:.2f}  ->  N >= {n_real:.1f}  (use N = {n_ceil})')
    if H_size == 10000 and abs(delta - 0.1) < 1e-9 and abs(eps - 0.1) < 1e-9:
        print('  matches the lecture: |H|=10000, delta=0.1, eps=0.1  ->  N >= 610.')

    eps_axis = np.linspace(0.02, 0.30, 80)
    n_vs_eps = [required_n(H_size, delta, ee) for ee in eps_axis]
    H_axis = np.logspace(1, 6, 60)
    n_vs_H = [required_n(hh, delta, eps) for hh in H_axis]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax1.plot(eps_axis, n_vs_eps, color=COLORS['accent'], lw=2.0)
    style_axes(ax1, title='Strong 1/eps^2 dependence', xlabel='tolerance eps', ylabel='required N')
    ax2.plot(H_axis, n_vs_H, color=COLORS['digimon'], lw=2.0)
    ax2.set_xscale('log')
    style_axes(ax2, title='Mild log dependence on |H|', xlabel='|H|  (log scale)', ylabel='required N')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    fig_sample_complexity = fig

interact(_draw_sample_complexity,
         H_size=IntSlider(min=10, max=100000, step=10, value=10000, description='|H|'),
         delta=FloatSlider(min=0.01, max=0.50, step=0.01, value=0.1, description='delta'),
         eps=FloatSlider(min=0.02, max=0.30, step=0.01, value=0.1, description='eps'))

In [ ]:
# === The bias-complexity tradeoff (answering C03's overfitting question) ===

def _subgrid_for_complexity(m, edge_range):
    '''m thresholds spanning EDGE_RANGE (deduped); falls back to arange if collapsed.'''
    lo, hi = edge_range
    m = int(max(1, m))
    g = np.unique(np.linspace(lo, hi, m).astype(int))
    if g.size < 1:
        g = np.arange(lo, hi + 1)
    return g

def complexity_curves(H_sizes, N, generator, T=15):
    '''For each candidate complexity m: ideal loss, expected gap, expected test loss.'''
    H_sizes = np.asarray(H_sizes)
    ideal = np.empty(H_sizes.size)
    gap = np.empty(H_sizes.size)
    for i, m in enumerate(H_sizes):
        sub = _subgrid_for_complexity(int(m), EDGE_RANGE)
        lv_all = sweep_loss(sub, e_all, y_all)
        ideal_m = float(lv_all.min())
        gaps = np.empty(T)
        for t in range(T):
            g = np.random.default_rng(SEED + int(N) + int(m) + t)
            idx = g.choice(len(e_all), size=int(N), replace=True)
            lv_tr = sweep_loss(sub, e_all[idx], y_all[idx])
            h_tr = int(sub[int(np.argmin(lv_tr))])
            gaps[t] = loss(h_tr, e_all, y_all) - ideal_m
        ideal[i] = ideal_m
        gap[i] = float(np.mean(gaps))
    return {'H_sizes': H_sizes, 'ideal_loss': ideal,
            'expected_gap': gap, 'expected_test_loss': ideal + gap}

H_sizes = np.unique(np.logspace(0.7, 4, 12).astype(int))
fig_complexity_tradeoff = None
best_complexity = None

def _draw_tradeoff(N):
    global fig_complexity_tradeoff, best_complexity
    c = complexity_curves(H_sizes, int(N), rng, T=15)
    best_complexity = int(c['H_sizes'][int(np.argmin(c['expected_test_loss']))])

    fig, ax = new_axes(figsize=(9, 4.6))
    ax.plot(c['H_sizes'], c['ideal_loss'], 'o-', color=COLORS['ideal'],
            label='ideal loss (lower bias)')
    ax.plot(c['H_sizes'], np.maximum(c['expected_gap'], 0.0), 'o-', color=COLORS['accent'],
            label='expected gap (higher variance)')
    ax.plot(c['H_sizes'], c['expected_test_loss'], 'o-', color=COLORS['digimon'], lw=2.4,
            label='expected test loss (sum)')
    ax.axvline(best_complexity, color='gray', ls='--', lw=1.4,
               label=f'best |H| ~ {best_complexity}')
    ax.set_xscale('log')
    style_axes(ax, title=f'Bias-complexity tradeoff (N={int(N)}): the U-shape',
               xlabel='model complexity |H|  (log scale)', ylabel='loss')
    ax.legend(loc='upper center', fontsize=9)
    plt.show()
    plt.close(fig)
    fig_complexity_tradeoff = fig
    print(f'At N={int(N)}: expected test loss is minimized near |H| ~ {best_complexity}.')
    print('Bigger |H| lowers bias but widens the gap. Deep learning aims for BOTH '
          'low bias and a small gap.')

interact(_draw_tradeoff,
         N=IntSlider(min=20, max=1000, step=20, value=200, description='N',
                     continuous_update=False))

## Recap: seven ideas, one story

1. **Threshold model & $|H|$.** A one-knob classifier $f_h(e)=\mathbb{1}[e \ge h]$; the hypothesis set $H=\{1,\dots,10000\}$, so $|H|=10000$ measures complexity.
2. **Loss = 0/1 error rate.** $L(h,D)=\frac1N\sum \mathbb{1}[f_h(x)\neq \hat y]$, a number in $[0,1]$ that changes as you move $h$.
3. **Ideal vs empirical.** $h_{all}$ minimizes loss on the universe (here $L(h_{all},D_{all})\approx 0.28$, with $h_{all}$ the value C06 printed); $h_{train}$ minimizes it on a sample. Their gap is what we pay for learning from data.
4. **Good training set.** If $\forall h:\ |L(h,D_{train})-L(h,D_{all})|\le\varepsilon$ with $\varepsilon=\delta/2$, the gap is $\le\delta$.
5. **The guarantee.** $P(D_{train}\text{ is bad}) \le |H|\cdot 2\exp(-2N\varepsilon^2)$ — distribution-free, model-agnostic.
6. **Sample complexity.** Invert it: $N \ge \log(2|H|/\delta)/(2\varepsilon^2)$; for $|H|=10000,\ \delta=0.1,\ \varepsilon=0.1$ this is $N \ge 610$.
7. **Bias–complexity tradeoff.** Larger $|H|$ lowers achievable loss but widens the gap → a U-shaped test loss with an optimal complexity.

### The takeaway
**Larger $N$ and smaller $|H|$ both make a sampled training set trustworthy** — but a small $|H|$ also raises the best loss you could ever reach. That tension is the whole game.

> 魚與熊掌可以兼得嗎? — *Can we have both the fish and the bear's paw?*
> The bound says small $|H|$ (good generalization) fights low bias. **Yes — Deep Learning** is the modern answer: huge, expressive models that nonetheless generalize.

### References
- Original course materials: *Machine Learning* — generalization, Hoeffding's inequality, and the |H|/N tradeoff (the Pokémon-vs-Digimon lecture).
- Hoeffding, W. (1963). *Probability inequalities for sums of bounded random variables.*
- Every figure above was produced from the synthetic universe built in this notebook; restart-and-run-all reproduces every number.